# RuSearchRank Phase 3 — frozen Colab GPU protocol

Supervised fine-tuning of `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` at revision `1427fd652930e4ba29e8149678df786c240d8825`. Run cells once, in order. Training and full final scoring are Colab-GPU-only.


In [ ]:
import os, platform, shutil, subprocess
from pathlib import Path

if platform.system() != 'Linux':
    raise RuntimeError('Phase 3 production protocol requires Linux Colab')
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('A CUDA GPU runtime is required')
MIN_FREE_GIB = 25
MIN_RAM_GIB = 12
if shutil.disk_usage('/content').free < MIN_FREE_GIB * 1024**3:
    raise RuntimeError('Insufficient free disk')
available_ram = os.sysconf('SC_AVPHYS_PAGES') * os.sysconf('SC_PAGE_SIZE')
if available_ram < MIN_RAM_GIB * 1024**3:
    raise RuntimeError('Insufficient system RAM')
print(gpu.stdout)


In [ ]:
import hashlib, json, sys

BRANCH = 'phase-3'
REPOSITORY_URL = 'https://github.com/kopanevk/ru-search-rank.git'
REPOSITORY = Path('/content/ru-search-rank')
ALLOW_OVERWRITE_PHASE3 = False
RESUME_TRAINING = True

def run_checked(command, *, cwd=REPOSITORY, stream=False):
    print('$', ' '.join(map(str, command)), flush=True)
    if stream:
        result = subprocess.run(command, cwd=cwd, text=True)
    else:
        result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
        print(result.stdout)
        if result.stderr:
            print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'command failed with return code {result.returncode}; review the complete log')
    return result

def training_options(run_id):
    if globals().get('FINETUNE_SMOKE_PASSED') is not True:
        raise RuntimeError('real smoke must pass before any finetune run')
    if ALLOW_OVERWRITE_PHASE3:
        return ['--overwrite']
    manifest = REPOSITORY / f'artifacts/models/{run_id}/run_manifest.json'
    return ['--resume'] if RESUME_TRAINING and manifest.is_file() else []

if not REPOSITORY.is_dir():
    run_checked(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY_URL, str(REPOSITORY)], cwd=Path('/content'))
branch = run_checked(['git', 'branch', '--show-current']).stdout.strip()
status = run_checked(['git', 'status', '--short']).stdout.strip()
if branch != BRANCH or status:
    raise RuntimeError(f'repository must be a clean {BRANCH} checkout')
run_checked(['git', 'pull', '--ff-only', 'origin', BRANCH], stream=True)


In [ ]:
import re
from datetime import datetime, timezone

TREC_EVAL_TAG = 'v9.0.8'
TREC_EVAL_COMMIT = 'd95ca64e14a47d763ae349fb65e6d8cde4141dbd'
TREC_EVAL_DIR = Path('/content/trec_eval')
run_checked(['apt-get', 'update'], cwd=Path('/content'), stream=True)
run_checked(['apt-get', 'install', '-y', 'openjdk-21-jdk-headless', 'build-essential'], cwd=Path('/content'), stream=True)
if TREC_EVAL_DIR.exists():
    shutil.rmtree(TREC_EVAL_DIR)
run_checked(['git', 'clone', '--depth', '1', '--branch', TREC_EVAL_TAG, 'https://github.com/usnistgov/trec_eval.git', str(TREC_EVAL_DIR)], cwd=Path('/content'))
head = run_checked(['git', 'rev-parse', 'HEAD'], cwd=TREC_EVAL_DIR).stdout.strip()
tagged = run_checked(['git', 'rev-list', '-n', '1', TREC_EVAL_TAG], cwd=TREC_EVAL_DIR).stdout.strip()
if head != tagged or head != TREC_EVAL_COMMIT:
    raise RuntimeError(f'unexpected trec_eval commit {head}')
run_checked(['git', 'diff', '--quiet', 'HEAD'], cwd=TREC_EVAL_DIR)
run_checked(['git', 'diff', '--cached', '--quiet'], cwd=TREC_EVAL_DIR)
makefile_sha256_before = hashlib.sha256((TREC_EVAL_DIR / 'Makefile').read_bytes()).hexdigest()
jobs = os.cpu_count() or 2
build_command = f'make -j{jobs}'
run_checked(['make', f'-j{jobs}'], cwd=TREC_EVAL_DIR, stream=True)
run_checked(['git', 'diff', '--quiet', 'HEAD'], cwd=TREC_EVAL_DIR)
run_checked(['git', 'diff', '--cached', '--quiet'], cwd=TREC_EVAL_DIR)
makefile_sha256_after = hashlib.sha256((TREC_EVAL_DIR / 'Makefile').read_bytes()).hexdigest()
if makefile_sha256_after != makefile_sha256_before:
    raise RuntimeError('upstream Makefile changed during build')
Path('/opt/bin').mkdir(parents=True, exist_ok=True)
run_checked(['install', '-m', '0755', str(TREC_EVAL_DIR / 'trec_eval'), '/opt/bin/trec_eval'], cwd=Path('/content'))
version_probe = run_checked(['/opt/bin/trec_eval', '-v'], cwd=Path('/content'))
if not re.search(r'\b9\.0\.7\b', version_probe.stdout or version_probe.stderr):
    raise RuntimeError('official v9.0.8 binary must report 9.0.7')
binary_sha256 = hashlib.sha256(Path('/opt/bin/trec_eval').read_bytes()).hexdigest()
compiler = run_checked(['cc', '--version'], cwd=Path('/content')).stdout.splitlines()[0]
provenance = {
    'source_repository': 'https://github.com/usnistgov/trec_eval.git',
    'source_tag': TREC_EVAL_TAG,
    'source_commit': head,
    'source_tree_clean': True,
    'fresh_checkout': True,
    'source_path': str(TREC_EVAL_DIR),
    'makefile_sha256': makefile_sha256_after,
    'binary_path': '/opt/bin/trec_eval',
    'binary_sha256': binary_sha256,
    'binary_reported_version': '9.0.7',
    'expected_release_version': '9.0.8',
    'known_upstream_version_string_mismatch': True,
    'build_command': build_command,
    'compiler': compiler,
    'built_at': datetime.now(timezone.utc).isoformat(),
}
provenance_path = REPOSITORY / 'artifacts/work/phase2/trec_eval_build_provenance.json'
provenance_path.parent.mkdir(parents=True, exist_ok=True)
provenance_path.write_text(json.dumps(provenance, indent=2) + '\n', encoding='utf-8')
print(json.dumps(provenance, indent=2))


In [ ]:
import venv
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f'Python 3.12 is required, got {sys.version}')
VENV_PATH = Path('/content/rusearchrank-phase3-venv')
venv.EnvBuilder(with_pip=True, clear=True).create(VENV_PATH)
RUN_PYTHON = str(VENV_PATH / 'bin/python')
run_checked([RUN_PYTHON, '-m', 'pip', 'install', '-U', 'pip'], cwd=REPOSITORY, stream=True)
run_checked([RUN_PYTHON, '-m', 'pip', 'install', '-e', '.'], cwd=REPOSITORY, stream=True)
run_checked([RUN_PYTHON, '-c', "from huggingface_hub import hf_hub_download; repo='cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'; rev='1427fd652930e4ba29e8149678df786c240d8825'; [hf_hub_download(repo_id=repo, filename=name, revision=rev) for name in ('tokenizer.json','tokenizer_config.json','special_tokens_map.json','sentencepiece.bpe.model')]"], cwd=REPOSITORY, stream=True)
run_checked([RUN_PYTHON, '-c', "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.version.cuda)"], cwd=REPOSITORY)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'pytest', '-q'], cwd=REPOSITORY, stream=True)
for validator in ('validate_phase1_notebook.py', 'validate_phase2_notebook.py', 'validate_phase3_notebook.py'):
    run_checked([RUN_PYTHON, f'scripts/{validator}'], cwd=REPOSITORY)
run_checked([RUN_PYTHON, '-c', "import platform, torch, transformers, tokenizers; print({'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'tokenizers': tokenizers.__version__, 'cuda': torch.version.cuda})"], cwd=REPOSITORY)


In [ ]:
from google.colab import drive, files
import zipfile
drive.mount('/content/drive')
PHASE1_ZIP = Path('/content/drive/MyDrive/rusearchrank_phase1_results.zip')
PHASE2_ZIP = Path('/content/drive/MyDrive/rusearchrank_phase2_results.zip')

def restore_archive(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        for name in names:
            member = Path(name)
            if member.is_absolute() or '..' in member.parts:
                raise RuntimeError(f'unsafe archive member: {name}')
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f'CRC failed for {bad_member}')
        archive.extractall(REPOSITORY)
    validated_manifests = 0
    for name in names:
        if not name.endswith('manifest.json'):
            continue
        candidate = json.loads((REPOSITORY / name).read_text(encoding='utf-8'))
        entries = candidate.get('artifacts', candidate.get('files'))
        if not isinstance(entries, list):
            continue
        for entry in entries:
            payload_path = REPOSITORY / entry['path']
            if not payload_path.is_file() or payload_path.stat().st_size != entry['size_bytes']:
                raise RuntimeError(f'restored payload size mismatch: {entry["path"]}')
            if hashlib.sha256(payload_path.read_bytes()).hexdigest() != entry['sha256']:
                raise RuntimeError(f'restored payload hash mismatch: {entry["path"]}')
        validated_manifests += 1
    if validated_manifests != 1:
        raise RuntimeError(f'expected one payload manifest in {path.name}')

restore_archive(PHASE1_ZIP)
restore_archive(PHASE2_ZIP)
print('Phase 1/2 artifacts restored')

run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'prepare-annotations', '--config', 'configs/retrieval.yaml', '--split', 'train'], cwd=REPOSITORY, stream=True)
print('Pinned train annotations materialized; final-evaluation annotations remain sealed')


In [ ]:
CONFIG = 'configs/finetune.yaml'
snapshot_code = "import json; from pathlib import Path; from rusearchrank.training_data import load_finetune_config, phase12_immutable_snapshot; c=load_finetune_config(Path('configs/finetune.yaml')); print(json.dumps(phase12_immutable_snapshot(c), sort_keys=True))"
def phase12_snapshot():
    return json.loads(run_checked([RUN_PYTHON, '-c', snapshot_code], cwd=REPOSITORY).stdout)
phase12_inputs = phase12_snapshot()
phase12_snapshot_path = REPOSITORY / 'artifacts/work/phase3/phase12_preselection_snapshot.json'
phase12_snapshot_path.parent.mkdir(parents=True, exist_ok=True)
phase12_snapshot_path.write_text(json.dumps(phase12_inputs, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(phase12_inputs, indent=2))
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'build-training-split', '--config', CONFIG] + (['--overwrite'] if ALLOW_OVERWRITE_PHASE3 else []), cwd=REPOSITORY, stream=True)
for regime in ('judged_only', 'weak_negatives', 'control_c1'):
    run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'build-training-pairs', '--config', CONFIG, '--regime', regime] + (['--overwrite'] if ALLOW_OVERWRITE_PHASE3 else []), cwd=REPOSITORY, stream=True)
verify_code = "import json; from pathlib import Path; from rusearchrank.training_data import load_finetune_config, verify_phase12_immutable; c=load_finetune_config(Path('configs/finetune.yaml')); expected=json.loads(Path('artifacts/work/phase3/phase12_preselection_snapshot.json').read_text()); verify_phase12_immutable(c, expected)"
run_checked([RUN_PYTHON, '-c', verify_code], cwd=REPOSITORY)
pairs_manifest = json.loads((REPOSITORY / 'reports/audit/pairs_manifest.json').read_text(encoding='utf-8'))
for regime in ('judged_only', 'weak_negatives', 'control_c1'):
    section = pairs_manifest['regimes'][regime]
    print(json.dumps({key: section[key] for key in ('regime_id', 'usable_query_count', 'judged_pairs_before_cap', 'judged_pairs_after_cap', 'weak_pairs_before_cap', 'weak_pairs_after_cap', 'leakage_audit', 'population_disclosure', 'weight_disclosure', 'heuristic_disclosure')}, ensure_ascii=False, indent=2))


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'validate-checkpoint', '--config', CONFIG, '--checkpoint', 'base'], cwd=REPOSITORY, stream=True)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'smoke-finetune', '--config', CONFIG, '--limit-pairs', '64'], cwd=REPOSITORY, stream=True)
smoke = json.loads((REPOSITORY / 'reports/audit/finetune_smoke.json').read_text())
if smoke.get('status') != 'PASS' or smoke.get('real_model_forward') is not True or smoke.get('fixture_only') is not False:
    raise RuntimeError('real smoke gate did not pass')
FINETUNE_SMOKE_PASSED = True
resource = json.loads((REPOSITORY / 'reports/audit/resource_report.json').read_text())
print(json.dumps({'smoke': smoke, 'resource_report': resource, 'estimated_training_time_range_seconds': resource['estimated_training_time_range_seconds']}, indent=2))


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'finetune', '--config', CONFIG, '--run-id', 'C1'] + training_options('C1'), cwd=REPOSITORY, stream=True)
control = json.loads((REPOSITORY / 'reports/audit/control_c1.json').read_text())
if control['status'] in ('FAIL', 'BLOCKED_FOR_REVIEW'):
    raise RuntimeError(f"C1 stopped the protocol: {control['status']} — {control['reason']}")


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'finetune', '--config', CONFIG, '--run-id', 'A1'] + training_options('A1'), cwd=REPOSITORY, stream=True)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'finetune', '--config', CONFIG, '--run-id', 'A2'] + training_options('A2'), cwd=REPOSITORY, stream=True)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'finetune', '--config', CONFIG, '--run-id', 'B1'] + training_options('B1'), cwd=REPOSITORY, stream=True)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'select-checkpoint', '--config', CONFIG] + (['--overwrite'] if ALLOW_OVERWRITE_PHASE3 else []), cwd=REPOSITORY, stream=True)
selection = json.loads((REPOSITORY / 'reports/audit/checkpoint_selection.json').read_text())
ab_report = json.loads((REPOSITORY / 'reports/metrics/validation_ab_comparison.json').read_text())
print(json.dumps({'candidates': selection['candidates'], 'best_finetuned_checkpoint': selection['best_finetuned_checkpoint'], 'production_system': selection['production_system'], 'zero_shot_won': selection['zero_shot_won'], 'exploratory_post_selection_ab': ab_report}, ensure_ascii=False, indent=2))


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'prepare-dev-evaluation', '--config', CONFIG], cwd=REPOSITORY, stream=True)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'score-finetuned', '--config', CONFIG] + (['--overwrite'] if ALLOW_OVERWRITE_PHASE3 else []), cwd=REPOSITORY, stream=True)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'evaluate-phase3', '--config', CONFIG] + (['--overwrite'] if ALLOW_OVERWRITE_PHASE3 else []), cwd=REPOSITORY, stream=True)
comparison = json.loads((REPOSITORY / 'reports/metrics/dev_three_way_comparison.json').read_text())
print(json.dumps({'systems': comparison['systems'], 'pipeline_status': comparison['pipeline_status'], 'ml_outcome': comparison['ml_outcome']}, ensure_ascii=False, indent=2))


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'package-phase3', '--config', CONFIG] + (['--overwrite'] if ALLOW_OVERWRITE_PHASE3 else []), cwd=REPOSITORY, stream=True)
result_zip = REPOSITORY / 'artifacts/rusearchrank_phase3_results.zip'
selection = json.loads((REPOSITORY / 'reports/audit/checkpoint_selection.json').read_text())
model_zip = REPOSITORY / f"artifacts/rusearchrank_phase3_model_{selection['best_finetuned_checkpoint']['run_id']}.zip"
def streaming_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
for archive_path in (result_zip, model_zip):
    with zipfile.ZipFile(archive_path) as archive:
        if archive.testzip() is not None:
            raise RuntimeError(f'CRC failed: {archive_path}')
        print({'name': archive_path.name, 'size_bytes': archive_path.stat().st_size, 'sha256': streaming_sha256(archive_path), 'members': archive.namelist()})
    shutil.copy2(archive_path, Path('/content/drive/MyDrive') / archive_path.name)
    files.download(str(archive_path))
